# Policy Copilot — Data exploration (F1.1 + F1.2)

逐个 block 运行，理解 `src/policy_copilot/data.py` 做了什么。

**如何运行：**
- VS Code：打开本文件 → 右上角选内核 `.venv` (policy-copilot)。
- 终端：`uv run jupyter lab` → 打开 `notebooks/explore_data.ipynb`。

每个代码块按 **Shift+Enter** 运行。首次运行 `load_rows()` 会联网下载数据（小，几秒）。

## 1. 导入我们自己写的函数

In [ ]:
from collections import Counter

from policy_copilot.data import (
    CATEGORY_MATH,
    CATEGORY_NOT_FOUND,
    DATASET_ID,
    build_corpus,
    category_counts,
    load_rows,
)

DATASET_ID

## 2. 加载 200 条数据

In [ ]:
rows = load_rows()
len(rows)

## 3. 看一条完整数据

每行有：`query`(问题)、`answer`(标准答案)、`context`(原文)、`category`(类别)。

In [ ]:
rows[0]

## 4. 类别分布

真实标签：`core` / `not_found_classification` / `boolean` / `math_basic` / `complex_qa` / `summary`。

In [ ]:
category_counts(rows)

## 5. 去重成文档库（F1.2）

200 条问答 → 51 份唯一文档。这 51 份就是后面要被检索的“知识库”。

In [ ]:
corpus = build_corpus(rows)
len(corpus)

In [ ]:
doc = corpus[0]
print(doc.doc_id)
print(doc.text[:300], "...")

## 6. 看两个关键类别的例子

`not_found`（应当拒答）和 `math`（数字要逐字）是治理评估的重点。

In [ ]:
nf = [r for r in rows if r["category"] == CATEGORY_NOT_FOUND]
print("not_found 样本数:", len(nf))
print("Q:", nf[0]["query"])
print("A:", nf[0]["answer"])  # 期望：类似 'Not Found'

In [ ]:
mm = [r for r in rows if r["category"] == CATEGORY_MATH]
print("Q:", mm[0]["query"])
print("A:", mm[0]["answer"])  # 数字答案，必须来自原文

## 7. “双用途 context”演示

同一份 context 会服务多个不同问题 —— 这就是为什么去重后只剩 51 份。

In [ ]:
ctx_counts = Counter(r["context"] for r in rows)
top_ctx, n = ctx_counts.most_common(1)[0]
print(f"这份 context 被 {n} 个不同问题共用:\n")
for r in rows:
    if r["context"] == top_ctx:
        print(" -", r["query"])